# Carpet WaveToy Thorns

## Author: Zach Etienne

To run the whole notebook, click the `>>` toolbar button and choose
**Restart Kernel and Run All Cells...**.

This notebook generates a Carpet-compatible WaveToy thorn family, inspects the
generated CCL metadata and representative source files, and validates that the
evolution, initial-data, and diagnostic thorns contain the expected schedule
entries and generated C functions.

**Notebook Status:** Validated for generated Carpet-compatible thorn-family
content. External Einstein Toolkit/Carpet build/run not executed by this
notebook.

**Validation Notes:** The validation section checks the generated `WaveToyNRPy`,
`IDWaveToyNRPy`, and `diagWaveToyNRPy` thorn inventories, CCL metadata,
schedule entries, build metadata, and representative complete source files. A
full runtime test requires an Einstein Toolkit checkout with Carpet.

Navigation:
[Index](../index.ipynb) |
Previous: [ETLegacy Core Infrastructure](etlegacy_core_infrastructure.ipynb) |
Next: [Code-Writing Path Choice Guide](backend_choice_guide.ipynb)

### Required Reading

- [ETLegacy Core Infrastructure](etlegacy_core_infrastructure.ipynb): defines
  CCL files, schedule bins, thorns, and generated source metadata.
- [Start-to-Finish Cartesian Wave Project][cart]: gives the wave-equation
  project context behind this WaveToy thorn family.

### NRPy Source Code

NRPy is used here as a pip-installed package. The import paths below identify
the installed modules used by the notebook; [NRPy upstream source][nrpy-source]
is the source browser for those modules.

- `nrpy.examples.carpet_wavetoy_thorns`: command-line generator for the thorn
  family used in this notebook.
- `nrpy.infrastructures.ETLegacy.interface_ccl`: writes variable and inherited
  capability declarations.
- `nrpy.infrastructures.ETLegacy.param_ccl`: writes runtime parameter
  declarations.
- `nrpy.infrastructures.ETLegacy.schedule_ccl`: writes schedule metadata.
- `nrpy.infrastructures.ETLegacy.make_code_defn`: writes the source-file list
  for each thorn.

[cart]: ../3-wave_equation/start_to_finish_wave_cartesian.ipynb
[nrpy-source]: https://github.com/nrpy/nrpy

# Table of Contents

1. [Words for This Notebook](#Words-for-This-Notebook)
1. [Workspace and Generated Files](#Workspace-and-Generated-Files)
1. [Step 1: Prepare the Carpet Thorn Workspace](#Step-1:-Prepare-the-Carpet-Thorn-Workspace)
1. [Step 2: Generate Carpet-Compatible Thorns](#Step-2:-Generate-Carpet-Compatible-Thorns)
1. [Step 3: Catalog the Thorn Family](#Step-3:-Catalog-the-Thorn-Family)
1. [Step 4: Inspect Evolution Thorn Metadata](#Step-4:-Inspect-Evolution-Thorn-Metadata)
1. [Step 5: Inspect Initial-Data Thorn Metadata](#Step-5:-Inspect-Initial-Data-Thorn-Metadata)
1. [Step 6: Inspect Diagnostics Thorn Metadata](#Step-6:-Inspect-Diagnostics-Thorn-Metadata)
1. [Step 7: Inspect a Complete Generated RHS Source][step7]
1. [Step 8: External Build and Run Commands](#Step-8:-External-Build-and-Run-Commands)
1. [Validation Check](#Validation-Check)
1. [What Next?](#What-Next?)

[step7]: #Step-7:-Inspect-a-Complete-Generated-RHS-Source

# Words for This Notebook
### [Back to [top](#Table-of-Contents)]

- **Einstein Toolkit:** a framework that builds and runs components called
  thorns.
- **Carpet:** an Einstein Toolkit mesh driver used by older Toolkit examples.
- **WaveToy:** a simple wave-equation thorn family used for Toolkit tutorials.
- **Thorn:** a component directory containing CCL metadata and source files.
- **Evolution thorn:** the thorn that owns evolved fields and RHS schedules;
  here it is `WaveToyNRPy`.
- **Initial-data thorn:** the thorn that fills fields before evolution starts;
  here it is `IDWaveToyNRPy`.
- **Diagnostic thorn:** the thorn that writes or computes comparison data; here
  it is `diagWaveToyNRPy`.
- **CCL:** Cactus Configuration Language, the metadata language used by
  Einstein Toolkit thorns.
- **`interface.ccl`:** the CCL file that declares variables and inherited
  capabilities.
- **`param.ccl`:** the CCL file that declares runtime parameters.
- **`schedule.ccl`:** the CCL file that says when generated functions run.
- **`make.code.defn`:** the source-file list consumed by the Toolkit build.
- **Schedule bin:** a named runtime phase such as `MoL_CalcRHS`,
  `CCTK_INITIAL`, or `CCTK_ANALYSIS`.
- **Method of Lines:** the time-integration pattern that calls the RHS in
  `MoL_CalcRHS`.
- **`uu` and `vv`:** the WaveToy evolved fields.
- **`uu_exact` and `vv_exact`:** auxiliary fields used by the diagnostic thorn
  to store exact-solution values.
- **Diagnostic:** generated output or metadata used to compare numerical and
  exact wave fields.

# Workspace and Generated Files
### [Back to [top](#Table-of-Contents)]

This notebook owns and recreates `project/carpet_wavetoy_thorns/` under the
`5-infrastructures` directory. Each run removes only that owned directory,
regenerates the thorn family, and moves the generator's output into:

`project/carpet_wavetoy_thorns/et_wavetoy`

The thorn roles are:

| Thorn | Purpose | Where used | What to inspect |
| --- | --- | --- | --- |
| `WaveToyNRPy` | Evolves wave fields | RHS workflow | CCL and RHS source |
| `IDWaveToyNRPy` | Sets initial data | initialization | exact-solution setup |
| `diagWaveToyNRPy` | Writes diagnostics | analysis | schedule and source |

# Step 1: Prepare the Carpet Thorn Workspace
### [Back to [top](#Table-of-Contents)]

This setup cell defines the stable project path and helper functions used for
inventory and validation. Skim the helper definitions on a first pass.

In [1]:
from pathlib import Path
import os
import re
import shutil
import subprocess
import sys


def find_notebook_dir():
    cwd = Path.cwd().resolve()
    if cwd.name == "5-infrastructures" and (cwd / "backend_choice_guide.ipynb").is_file():
        return cwd
    candidate = cwd / "5-infrastructures"
    if (candidate / "backend_choice_guide.ipynb").is_file():
        return candidate.resolve()
    raise RuntimeError("Run this notebook from the repository root or 5-infrastructures/.")


NOTEBOOK_DIR = find_notebook_dir()

The command helper exposes failures and the inventory helper records the exact
generated file set.

In [2]:
def run_command(args, cwd, timeout=300):
    result = subprocess.run(
        args,
        cwd=cwd,
        env=os.environ.copy(),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
        timeout=timeout,
    )
    if result.returncode != 0:
        print(result.stdout)
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {args}")
    return result.stdout


def file_inventory(root):
    return sorted(str(path.relative_to(root)) for path in root.rglob("*") if path.is_file())


def print_check(name, passed, detail):
    status = "PASS" if passed else "FAIL"
    print(f"{status:4} | {name} | {detail}")

Now create the owned workspace and print the project path before generation.

In [3]:
WORKSPACE = NOTEBOOK_DIR / "project" / "carpet_wavetoy_thorns"
PROJECT_DIR = WORKSPACE / "et_wavetoy"
GENERATOR_OUTPUT_DIR = WORKSPACE / "project" / "et_wavetoy"

if WORKSPACE.exists():
    shutil.rmtree(WORKSPACE)
WORKSPACE.mkdir(parents=True)

print("workspace:", WORKSPACE.relative_to(NOTEBOOK_DIR))
print("project path:", PROJECT_DIR.relative_to(NOTEBOOK_DIR))

workspace: project/carpet_wavetoy_thorns
project path: project/carpet_wavetoy_thorns/et_wavetoy


# Step 2: Generate Carpet-Compatible Thorns
### [Back to [top](#Table-of-Contents)]

The trusted generator is `python -m nrpy.examples.carpet_wavetoy_thorns`.
Like other NRPy examples, it writes under a nested `project/` directory
relative to its working directory; this notebook moves the generated thorn
family into the stable project path printed above.

In [4]:
command = [sys.executable, "-m", "nrpy.examples.carpet_wavetoy_thorns"]
print("generator command:", "python -m nrpy.examples.carpet_wavetoy_thorns")
generator_output = run_command(command, cwd=WORKSPACE, timeout=300)
clean_output = re.sub(r"\x1b\[[0-?]*[ -/]*[@-~]", "", generator_output)
print(clean_output.replace(str(NOTEBOOK_DIR), ".").strip())

if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
shutil.move(str(GENERATOR_OUTPUT_DIR), str(PROJECT_DIR))
shutil.rmtree(WORKSPACE / "project", ignore_errors=True)

print("project generated:", PROJECT_DIR.relative_to(NOTEBOOK_DIR))

generator command: python -m nrpy.examples.carpet_wavetoy_thorns


In 0.198s, worker completed task 'register_CFunction_exact_solution_all_points'
In 0.220s, worker completed task 'register_CFunction_rhs_eval'
In 0.258s, worker completed task 'register_CFunction_exact_solution_all_points'
Checking ./project/carpet_wavetoy_thorns/project/et_wavetoy/WaveToyNRPy/schedule.ccl...[written]
Checking ./project/carpet_wavetoy_thorns/project/et_wavetoy/WaveToyNRPy/interface.ccl...[written]
Checking ./project/carpet_wavetoy_thorns/project/et_wavetoy/WaveToyNRPy/param.ccl...[written]
Checking ./project/carpet_wavetoy_thorns/project/et_wavetoy/IDWaveToyNRPy/schedule.ccl...[written]
Checking ./project/carpet_wavetoy_thorns/project/et_wavetoy/IDWaveToyNRPy/interface.ccl...[written]
Checking ./project/carpet_wavetoy_thorns/project/et_wavetoy/IDWaveToyNRPy/param.ccl...[written]
Checking ./project/carpet_wavetoy_thorns/project/et_wavetoy/diagWaveToyNRPy/schedule.ccl...[written]
Checking ./project/carpet_wavetoy_thorns/project/et_wavetoy/diagWaveToyNRPy/interface.ccl...

The generator messages identify which files were written. The complete
inventory in the next section is the required evidence.

# Step 3: Catalog the Thorn Family
### [Back to [top](#Table-of-Contents)]

The complete inventory should contain three thorns: evolution, initial data,
and diagnostics.

In [5]:
EXPECTED_THORN_INVENTORY = [
    "IDWaveToyNRPy/interface.ccl",
    "IDWaveToyNRPy/param.ccl",
    "IDWaveToyNRPy/schedule.ccl",
    "IDWaveToyNRPy/src/IDWaveToyNRPy_exact_solution_all_points.c",
    "IDWaveToyNRPy/src/make.code.defn",
    "WaveToyNRPy/interface.ccl",
    "WaveToyNRPy/param.ccl",
    "WaveToyNRPy/schedule.ccl",
    "WaveToyNRPy/src/WaveToyNRPy_MoL_registration.c",
    "WaveToyNRPy/src/WaveToyNRPy_Symmetry_registration_oldCartGrid3D.c",
    "WaveToyNRPy/src/WaveToyNRPy_rhs_eval.c",
    "WaveToyNRPy/src/WaveToyNRPy_specify_Driver_BoundaryConditions.c",
    "WaveToyNRPy/src/WaveToyNRPy_specify_NewRad_BoundaryConditions_parameters.c",
    "WaveToyNRPy/src/WaveToyNRPy_specify_aux_BoundaryConditions.c",
    "WaveToyNRPy/src/WaveToyNRPy_specify_evol_BoundaryConditions.c",
    "WaveToyNRPy/src/WaveToyNRPy_zero_rhss.c",
    "WaveToyNRPy/src/make.code.defn",
    "WaveToyNRPy/src/simd/simd_intrinsics.h",
    "diagWaveToyNRPy/interface.ccl",
    "diagWaveToyNRPy/param.ccl",
    "diagWaveToyNRPy/schedule.ccl",
    "diagWaveToyNRPy/src/diagWaveToyNRPy_exact_solution_all_points.c",
    "diagWaveToyNRPy/src/make.code.defn",
]

thorn_inventory = file_inventory(PROJECT_DIR)
for relative_path in thorn_inventory:
    print(relative_path)

IDWaveToyNRPy/interface.ccl
IDWaveToyNRPy/param.ccl
IDWaveToyNRPy/schedule.ccl
IDWaveToyNRPy/src/IDWaveToyNRPy_exact_solution_all_points.c
IDWaveToyNRPy/src/make.code.defn
WaveToyNRPy/interface.ccl
WaveToyNRPy/param.ccl
WaveToyNRPy/schedule.ccl
WaveToyNRPy/src/WaveToyNRPy_MoL_registration.c
WaveToyNRPy/src/WaveToyNRPy_Symmetry_registration_oldCartGrid3D.c
WaveToyNRPy/src/WaveToyNRPy_rhs_eval.c
WaveToyNRPy/src/WaveToyNRPy_specify_Driver_BoundaryConditions.c
WaveToyNRPy/src/WaveToyNRPy_specify_NewRad_BoundaryConditions_parameters.c
WaveToyNRPy/src/WaveToyNRPy_specify_aux_BoundaryConditions.c
WaveToyNRPy/src/WaveToyNRPy_specify_evol_BoundaryConditions.c
WaveToyNRPy/src/WaveToyNRPy_zero_rhss.c
WaveToyNRPy/src/make.code.defn
WaveToyNRPy/src/simd/simd_intrinsics.h
diagWaveToyNRPy/interface.ccl
diagWaveToyNRPy/param.ccl
diagWaveToyNRPy/schedule.ccl
diagWaveToyNRPy/src/diagWaveToyNRPy_exact_solution_all_points.c
diagWaveToyNRPy/src/make.code.defn


The complete inventory shows the three-thorn split and the representative C
source files used by validation.

# Step 4: Inspect Evolution Thorn Metadata
### [Back to [top](#Table-of-Contents)]

`WaveToyNRPy` owns the evolved fields, RHS fields, exact-solution auxiliary
fields, and Method-of-Lines schedule entries. Inspect the variable groups and
RHS schedule.

In [6]:
evolution_interface_text = (PROJECT_DIR / "WaveToyNRPy" / "interface.ccl").read_text(
    encoding="utf-8", errors="replace"
)
evolution_schedule_text = (PROJECT_DIR / "WaveToyNRPy" / "schedule.ccl").read_text(
    encoding="utf-8", errors="replace"
)
print("--- WaveToyNRPy/interface.ccl ---")
print(evolution_interface_text)
print("--- WaveToyNRPy/schedule.ccl ---")
print(evolution_schedule_text)

--- WaveToyNRPy/interface.ccl ---

# This interface.ccl file was automatically generated by NRPy.
#   You are advised against modifying it directly; instead
#   modify the Python code that generates it.

# With "implements", we give our thorn its unique name.
implements: WaveToyNRPy

# By "inheriting" other thorns, we tell the Toolkit that we
#   will rely on variables/function that exist within those
#   functions.
inherits: Boundary Grid MethodofLines

# Needed functions and #include's:
USES INCLUDE: Symmetry.h
USES INCLUDE: Boundary.h


# Needed Method of Lines function
CCTK_INT FUNCTION MoLRegisterEvolvedGroup(CCTK_INT IN EvolvedIndex, CCTK_INT IN RHSIndex)
REQUIRES FUNCTION MoLRegisterEvolvedGroup

# Needed Boundary Conditions function
CCTK_INT FUNCTION GetBoundarySpecification(CCTK_INT IN size, CCTK_INT OUT ARRAY nboundaryzones, CCTK_INT OUT ARRAY is_internal, CCTK_INT OUT ARRAY is_staggered, CCTK_INT OUT ARRAY shiftout)
USES FUNCTION GetBoundarySpecification

CCTK_INT FUNCTION S

The evolution thorn declares `uu`, `vv`, their RHS fields, and exact-solution
auxiliary fields. Its schedule places the RHS in `MoL_CalcRHS`.

# Step 5: Inspect Initial-Data Thorn Metadata
### [Back to [top](#Table-of-Contents)]

`IDWaveToyNRPy` inherits the evolution thorn's gridfunctions and schedules its
exact-solution function at `CCTK_INITIAL`.

In [7]:
id_interface_text = (PROJECT_DIR / "IDWaveToyNRPy" / "interface.ccl").read_text(
    encoding="utf-8", errors="replace"
)
id_schedule_text = (PROJECT_DIR / "IDWaveToyNRPy" / "schedule.ccl").read_text(
    encoding="utf-8", errors="replace"
)
print("--- IDWaveToyNRPy/interface.ccl ---")
print(id_interface_text)
print("--- IDWaveToyNRPy/schedule.ccl ---")
print(id_schedule_text)

--- IDWaveToyNRPy/interface.ccl ---

# This interface.ccl file was automatically generated by NRPy.
#   You are advised against modifying it directly; instead
#   modify the Python code that generates it.

# With "implements", we give our thorn its unique name.
implements: IDWaveToyNRPy

# By "inheriting" other thorns, we tell the Toolkit that we
#   will rely on variables/function that exist within those
#   functions.
inherits: Grid WaveToyNRPy # WaveToyNRPy provides all gridfunctions.

# Needed functions and #include's:


# FIXME: add info for symmetry conditions:
#    https://einsteintoolkit.org/thornguide/CactusBase/SymBase/documentation.html

# Tell the Toolkit that we want all gridfunctions
#    to be visible to other thorns by using
#    the keyword "public". Note that declaring these
#    gridfunctions *does not* allocate memory for them;
#    that is done by the schedule.ccl file.

public:

--- IDWaveToyNRPy/schedule.ccl ---
# This schedule.ccl file was automatically generate

The initial-data thorn writes `WaveToyNRPy::uuGF` and `WaveToyNRPy::vvGF`
before time evolution starts.

# Step 6: Inspect Diagnostics Thorn Metadata
### [Back to [top](#Table-of-Contents)]

`diagWaveToyNRPy` computes exact-solution auxiliary fields during the analysis
schedule. Inspect the inherited evolution thorn and the `CCTK_ANALYSIS`
schedule entry.

In [8]:
diag_interface_text = (PROJECT_DIR / "diagWaveToyNRPy" / "interface.ccl").read_text(
    encoding="utf-8", errors="replace"
)
diag_schedule_text = (PROJECT_DIR / "diagWaveToyNRPy" / "schedule.ccl").read_text(
    encoding="utf-8", errors="replace"
)
print("--- diagWaveToyNRPy/interface.ccl ---")
print(diag_interface_text)
print("--- diagWaveToyNRPy/schedule.ccl ---")
print(diag_schedule_text)

--- diagWaveToyNRPy/interface.ccl ---

# This interface.ccl file was automatically generated by NRPy.
#   You are advised against modifying it directly; instead
#   modify the Python code that generates it.

# With "implements", we give our thorn its unique name.
implements: diagWaveToyNRPy

# By "inheriting" other thorns, we tell the Toolkit that we
#   will rely on variables/function that exist within those
#   functions.
inherits: Grid WaveToyNRPy # WaveToyNRPy provides all gridfunctions.

# Needed functions and #include's:


# FIXME: add info for symmetry conditions:
#    https://einsteintoolkit.org/thornguide/CactusBase/SymBase/documentation.html

# Tell the Toolkit that we want all gridfunctions
#    to be visible to other thorns by using
#    the keyword "public". Note that declaring these
#    gridfunctions *does not* allocate memory for them;
#    that is done by the schedule.ccl file.

public:

--- diagWaveToyNRPy/schedule.ccl ---
# This schedule.ccl file was automatically ge

The diagnostic thorn writes `uu_exactGF` and `vv_exactGF` in the analysis
schedule, so runtime diagnostics can compare numerical and exact fields.

# Step 7: Inspect a Complete Generated RHS Source
### [Back to [top](#Table-of-Contents)]

The full RHS source is the representative generated C function for the thorn
family. Inspect the scheduled function name, the `wavespeed` parameter read,
stencil reads, and writes to `uu_rhsGF` and `vv_rhsGF`.

In [9]:
rhs_source_text = (
    PROJECT_DIR / "WaveToyNRPy" / "src" / "WaveToyNRPy_rhs_eval.c"
).read_text(encoding="utf-8", errors="replace")
print(rhs_source_text)

#include "cctk.h"
#include "cctk_Arguments.h"
#include "cctk_Parameters.h"
#include "math.h"
#include "simd/simd_intrinsics.h"

/**
 * Set RHSs for wave equation.
 */
void WaveToyNRPy_rhs_eval(CCTK_ARGUMENTS) {
  DECLARE_CCTK_ARGUMENTS_WaveToyNRPy_rhs_eval;
  const CCTK_REAL *restrict NOSIMDwavespeed = CCTK_ParameterGet("wavespeed", "IDWaveToyNRPy", NULL); // IDWaveToyNRPy::wavespeed
  const REAL_SIMD_ARRAY wavespeed = ConstSIMD(*NOSIMDwavespeed);                                     // IDWaveToyNRPy::wavespeed
  const CCTK_REAL NOSIMDinvdxx0 = 1.0 / CCTK_DELTA_SPACE(0);
  const REAL_SIMD_ARRAY invdxx0 = ConstSIMD(NOSIMDinvdxx0);
  const CCTK_REAL NOSIMDinvdxx1 = 1.0 / CCTK_DELTA_SPACE(1);
  const REAL_SIMD_ARRAY invdxx1 = ConstSIMD(NOSIMDinvdxx1);
  const CCTK_REAL NOSIMDinvdxx2 = 1.0 / CCTK_DELTA_SPACE(2);
  const REAL_SIMD_ARRAY invdxx2 = ConstSIMD(NOSIMDinvdxx2);
#pragma omp parallel for
  for (int i2 = cctk_nghostzones[2]; i2 < cctk_lsh[2] - cctk_nghostzones[2]; i2++) {
    for (in

The RHS source shows how the WaveToy equations become generated ETLegacy C
code with memory reads, finite-difference expressions, and RHS writes.

# Step 8: External Build and Run Commands
### [Back to [top](#Table-of-Contents)]

A full build requires an external Einstein Toolkit checkout with Carpet. The
commands below are terminal-ready templates; replace `/path/to/Cactus` and
`/path/to/repo` with local paths. They are not executed by this notebook and
are not counted as validation evidence.

```bash
cd /path/to/Cactus
mkdir -p arrangements/NRPyTutorial
REPO=/path/to/repo
SRC=$REPO/5-infrastructures/project/carpet_wavetoy_thorns/et_wavetoy
cp -R $SRC/WaveToyNRPy arrangements/NRPyTutorial/
cp -R $SRC/IDWaveToyNRPy arrangements/NRPyTutorial/
cp -R $SRC/diagWaveToyNRPy arrangements/NRPyTutorial/
printf '%s\n' \
  NRPyTutorial/WaveToyNRPy \
  NRPyTutorial/IDWaveToyNRPy \
  NRPyTutorial/diagWaveToyNRPy > thornlists/nrpy_tutorial_wavetoy.th
./simfactory/bin/sim build -j2 --thornlist thornlists/nrpy_tutorial_wavetoy.th
```

# Validation Check
### [Back to [top](#Table-of-Contents)]

Trusted source: the NRPy Carpet WaveToy generator and ETLegacy writers linked
above. Newly checked output: the generated thorn family in this notebook run.
The validation checks generated content only; it does not claim an external
Einstein Toolkit runtime test.

In [10]:
wave_make_text = (PROJECT_DIR / "WaveToyNRPy" / "src" / "make.code.defn").read_text()
diag_make_text = (PROJECT_DIR / "diagWaveToyNRPy" / "src" / "make.code.defn").read_text()

validation_specs = [
    ("inventory", thorn_inventory == EXPECTED_THORN_INVENTORY, "exact file list"),
    ("WaveToy implements", "implements: WaveToyNRPy" in evolution_interface_text),
    ("WaveToy inherits", "inherits: Boundary Grid MethodofLines" in evolution_interface_text),
    ("WaveToy evolved", "uuGF, vvGF" in evolution_interface_text),
    ("WaveToy rhs", "uu_rhsGF, vv_rhsGF" in evolution_interface_text),
    ("WaveToy exact", "uu_exactGF, vv_exactGF" in evolution_interface_text),
    (
        "WaveToy RHS schedule",
        "schedule WaveToyNRPy_rhs_eval in MoL_CalcRHS as rhs_eval"
        in evolution_schedule_text,
    ),
    (
        "WaveToy MoL schedule",
        "schedule WaveToyNRPy_MoL_registration in MoL_Register"
        in evolution_schedule_text,
    ),
    (
        "WaveToy ApplyBCs",
        "schedule GROUP ApplyBCs as WaveToyNRPy_evol_ApplyBCs" in evolution_schedule_text,
    ),
    ("ID implements", "implements: IDWaveToyNRPy" in id_interface_text),
    ("ID inherits", "inherits: Grid WaveToyNRPy" in id_interface_text),
    (
        "ID schedule",
        "schedule IDWaveToyNRPy_exact_solution_all_points IN CCTK_INITIAL"
        in id_schedule_text,
    ),
    ("ID writes uu", "WRITES: WaveToyNRPy::uuGF(Everywhere)" in id_schedule_text),
    ("ID writes vv", "WRITES: WaveToyNRPy::vvGF(Everywhere)" in id_schedule_text),
    ("diag implements", "implements: diagWaveToyNRPy" in diag_interface_text),
    ("diag inherits", "inherits: Grid WaveToyNRPy" in diag_interface_text),
    (
        "diag schedule",
        "schedule diagWaveToyNRPy_exact_solution_all_points IN CCTK_ANALYSIS"
        in diag_schedule_text,
    ),
    ("diag writes uu_exact", "WRITES: WaveToyNRPy::uu_exactGF(Everywhere)" in diag_schedule_text),
    ("diag writes vv_exact", "WRITES: WaveToyNRPy::vv_exactGF(Everywhere)" in diag_schedule_text),
    ("RHS function", "void WaveToyNRPy_rhs_eval(CCTK_ARGUMENTS)" in rhs_source_text),
    ("RHS arguments", "DECLARE_CCTK_ARGUMENTS_WaveToyNRPy_rhs_eval" in rhs_source_text),
    ("RHS uu write", "WriteSIMD(&uu_rhsGF" in rhs_source_text),
    ("RHS vv write", "WriteSIMD(&vv_rhsGF" in rhs_source_text),
    ("WaveToy make RHS", "WaveToyNRPy_rhs_eval.c" in wave_make_text),
    ("diag make exact", "diagWaveToyNRPy_exact_solution_all_points.c" in diag_make_text),
]

The next cell prints one pass/fail row per generated-content check and raises
an error if any required file or substring is missing.

In [11]:
for check in validation_specs:
    name, passed = check[0], bool(check[1])
    detail = check[2] if len(check) > 2 else "required substring"
    print_check(name, passed, detail)

failed = [check[0] for check in validation_specs if not check[1]]
if failed:
    raise RuntimeError("Validation failed: " + ", ".join(failed))
print("validation passed: Carpet WaveToy thorn-family content matches contract")

PASS | inventory | exact file list
PASS | WaveToy implements | required substring
PASS | WaveToy inherits | required substring
PASS | WaveToy evolved | required substring
PASS | WaveToy rhs | required substring
PASS | WaveToy exact | required substring
PASS | WaveToy RHS schedule | required substring
PASS | WaveToy MoL schedule | required substring
PASS | WaveToy ApplyBCs | required substring
PASS | ID implements | required substring
PASS | ID inherits | required substring
PASS | ID schedule | required substring
PASS | ID writes uu | required substring
PASS | ID writes vv | required substring
PASS | diag implements | required substring
PASS | diag inherits | required substring
PASS | diag schedule | required substring
PASS | diag writes uu_exact | required substring
PASS | diag writes vv_exact | required substring
PASS | RHS function | required substring
PASS | RHS arguments | required substring
PASS | RHS uu write | required substring
PASS | RHS vv write | required substring
PASS | Wa

The validation proves that the generated thorn family has the expected
evolution, initial-data, and diagnostic metadata plus representative generated
source.

# What Next?

- [Code-Writing Path Choice Guide](backend_choice_guide.ipynb)
- [ETLegacy Core Infrastructure](etlegacy_core_infrastructure.ipynb)